# Using the pretrained EuclidFM encoder

EuclidFM produces one 128-dimensional CLS embedding per source from its available photometry. This notebook trains a quasar, star and galaxy classifier in two ways. First, it caches embeddings from the frozen encoder and trains a small classifier. Then it starts from the same checkpoint and fine-tunes the encoder and classifier together. Both use the same labelled split.

### Files

| Download | Local filename |
| --- | --- |
| Checkpoint from the 900-epoch run and training summary (see repository README) | photometry_transformer_best.pt |
| Labelled Platinum Sample (download link to be added) | EuclidFM_Platinum Sample.csv |
| Working Catalogue, 100k parent sample (download link to be added) | euclid_random_100k_even_3fields_all_features.csv |

From the EuclidFM repository, install the dependencies with **pip install -r requirements.txt jupyter**. Place the downloaded checkpoint and CSVs in the repository root, or set their paths below. Open this notebook from the repository root or its notebook directory.

See the repository README to reproduce pretraining with the existing Python script.

In [ ]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

cwd = Path.cwd()
repo = next((p for p in (cwd, cwd.parent) if (p / "scripts" / "run_training.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Open this notebook from the EuclidFM repository or its notebook directory.")
sys.path.insert(0, str(repo))

from config import Config
from data import build_measurement_dataset_arcsinh, find_measurements
from models import PhotometryTransformer
from run_inference import load_and_filter

CHECKPOINT = repo / "photometry_transformer_best.pt"
PLATINUM_CSV = repo / "EuclidFM_Platinum Sample.csv"
WORKING_CSV = repo / "euclid_random_100k_even_3fields_all_features.csv"

SEED = 42
BATCH_SIZE = 32
ROWS_PER_CLASS = 300      # Set to None to use every eligible labelled source.
FROZEN_EPOCHS = 10
FINETUNE_EPOCHS = 2
WORKING_PREVIEW_ROWS = 256  # Set to None to encode the full working catalogue.

CLASS_NAMES = {0: "Quasar", 1: "Star", 2: "Galaxy"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

for path in (CHECKPOINT, PLATINUM_CSV, WORKING_CSV):
    if not path.is_file():
        raise FileNotFoundError(f"Set the path to the downloaded file: {path}")
print(f"Device: {DEVICE}")

## Prepare the photometry and labelled splits

Use the full working catalogue to recover the checkpoint’s 94-channel layout. The encoder uses the original positions of flux and uncertainty pairs, so both catalogues must share the same layout. Missing pairs keep their positions and are masked. The repository applies the asinh transform to the fluxes and uncertainties.

The example applies the repository's quality cuts and excludes any Platinum object IDs present in the working catalogue before a stratified 70:15:15 split. Its scores are for this tutorial split.

In [ ]:
with TemporaryDirectory() as tmp:
    working = load_and_filter(WORKING_CSV, Path(tmp) / "working.csv")
    platinum = load_and_filter(PLATINUM_CSV, Path(tmp) / "platinum.csv")

measurements = find_measurements(working)
photometry_columns = list(dict.fromkeys(
    pair[kind] for pair in measurements for kind in ("flux", "error")
))

if not platinum["label"].isin(CLASS_NAMES).all():
    raise ValueError("Platinum labels must be 0 (quasar), 1 (star), or 2 (galaxy).")
if any(frame["OBJECT_ID"].isna().any() or not frame["OBJECT_ID"].is_unique
       for frame in (working, platinum)):
    raise ValueError("Both catalogues need one row per unique, non-missing OBJECT_ID.")

shared = platinum["OBJECT_ID"].isin(working["OBJECT_ID"])
print(f"Working sources after quality cuts: {len(working):,}")
print(f"Shared object IDs excluded from the labelled example: {shared.sum():,}")
platinum = platinum.loc[~shared]
if ROWS_PER_CLASS is not None:
    platinum = pd.concat(
        group.sample(n=min(len(group), ROWS_PER_CLASS), random_state=SEED)
        for _, group in platinum.groupby("label", sort=True)
    )

labels = torch.from_numpy(platinum["label"].to_numpy(dtype=np.int64, copy=True))
train_idx, held_out_idx = train_test_split(
    np.arange(len(labels)), test_size=0.30, stratify=labels.numpy(), random_state=SEED
)
val_idx, test_idx = train_test_split(
    held_out_idx, test_size=0.5, stratify=labels[held_out_idx].numpy(), random_state=SEED
)
print(f"Train / validation / test: {len(train_idx)} / {len(val_idx)} / {len(test_idx)}")


def prepare_photometry(frame):
    aligned = frame.reindex(columns=photometry_columns)
    values, valid, band, survey, kind, band_vocab, survey_vocab, type_vocab = (
        build_measurement_dataset_arcsinh(aligned, measurements)
    )
    arrays = (values, band, survey, kind, valid)
    return tuple(torch.from_numpy(array) for array in arrays), (
        len(band_vocab), len(survey_vocab), len(type_vocab)
    )


platinum_inputs, vocab_sizes = prepare_photometry(platinum)

## Load the checkpoint and expose the encoder

Use the best checkpoint from the 900-epoch run, saved at epoch 656. It uses the 128-dimensional encoder with global features disabled. The model's **encode** method uses every valid measurement to produce a normalized CLS vector and skips the reconstruction decoder.

In [ ]:
checkpoint = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
print(f"Checkpoint epoch: {checkpoint['epoch']}")
weights = checkpoint["model_state_dict"]
cfg = Config()
cfg.seq_len = len(measurements)

expected = (
    weights["embedding.pos_embed.weight"].shape[0] - 1,
    weights["embedding.band_embed.weight"].shape[0],
    weights["embedding.survey_embed.weight"].shape[0],
    weights["embedding.type_embed.weight"].shape[0],
)
if (cfg.seq_len, *vocab_sizes) != expected:
    raise ValueError(
        f"Catalogue layout {(cfg.seq_len, *vocab_sizes)} differs from checkpoint {expected}. "
        "Use the full working catalogue supplied with the model."
    )


def load_encoder():
    model = PhotometryTransformer(cfg, *vocab_sizes)
    model.load_state_dict(weights, strict=True)
    return model.to(DEVICE)


def make_loader(tensors, indices=None, shuffle=False):
    if indices is not None:
        tensors = tuple(tensor[indices] for tensor in tensors)
    return DataLoader(
        TensorDataset(*tensors), batch_size=BATCH_SIZE, shuffle=shuffle,
        generator=torch.Generator().manual_seed(SEED),
    )


@torch.no_grad()
def predict(predict_batch, loader):
    return torch.cat([
        predict_batch(*(tensor.to(DEVICE) for tensor in batch)).cpu() for batch in loader
    ])

## Train a classifier on frozen embeddings

Cache the CLS vectors once and fit the classifier on the training split. Choose the epoch by validation loss and report performance on the held-out test split.

In [ ]:
class SourceClassifier(nn.Module):
    def __init__(self, encoder=None):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(cfg.d_model, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, len(CLASS_NAMES)),
        )

    def forward(self, *inputs):
        embedding = self.encoder.encode(*inputs) if self.encoder is not None else inputs[0]
        return self.head(embedding)


def fit_classifier(model, inputs, epochs, optimizer):
    train_loader = make_loader((*inputs, labels), train_idx, shuffle=True)
    val_loader = make_loader(inputs, val_idx)
    criterion = nn.CrossEntropyLoss()
    best_loss = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        for *features, target in train_loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(*(tensor.to(DEVICE) for tensor in features))
            loss = criterion(logits, target.to(DEVICE))
            loss.backward()
            optimizer.step()

        val_loss = criterion(predict(model.eval(), val_loader), labels[val_idx]).item()
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = deepcopy(model.state_dict())
        print(f"Epoch {epoch:02d} | validation loss {val_loss:.4f}")

    model.load_state_dict(best_state)
    return model


def report_classifier(model, inputs):
    test_logits = predict(model.eval(), make_loader(inputs, test_idx))
    print(classification_report(
        labels[test_idx].numpy(), test_logits.argmax(dim=1).numpy(),
        labels=list(CLASS_NAMES), target_names=list(CLASS_NAMES.values()), zero_division=0,
    ))


frozen_encoder = load_encoder().requires_grad_(False).eval()
platinum_embeddings = predict(frozen_encoder.encode, make_loader(platinum_inputs))
print(f"Frozen embedding shape: {tuple(platinum_embeddings.shape)}")

torch.manual_seed(SEED)
frozen_classifier = SourceClassifier().to(DEVICE)
fit_classifier(
    frozen_classifier, (platinum_embeddings,), FROZEN_EPOCHS,
    torch.optim.AdamW(frozen_classifier.head.parameters(), lr=1e-3),
)
report_classifier(frozen_classifier, (platinum_embeddings,))

## Fine-tune the encoder and classifier

Start again from the published checkpoint. Feed photometry batches directly to the encoder so gradients can update it. Use a smaller learning rate for the pretrained layers than for the new classifier.

For another supervised task, keep the CLS encoder and adapt the target, output layer, loss and evaluation. For example, a scalar regression task uses one output and a regression loss such as Huber loss.

In [ ]:
finetune_encoder = load_encoder().requires_grad_(False)
for module in (finetune_encoder.embedding, finetune_encoder.encoder, finetune_encoder.encoder_norm):
    module.requires_grad_(True)
encoder_parameters = [p for p in finetune_encoder.parameters() if p.requires_grad]
torch.manual_seed(SEED)
finetuned_classifier = SourceClassifier(finetune_encoder).to(DEVICE)
optimizer = torch.optim.AdamW([
    {"params": encoder_parameters, "lr": 1e-5},
    {"params": finetuned_classifier.head.parameters(), "lr": 1e-3},
])
fit_classifier(finetuned_classifier, platinum_inputs, FINETUNE_EPOCHS, optimizer)
report_classifier(finetuned_classifier, platinum_inputs)

## Apply either classifier to the working catalogue

Keep object IDs alongside the predictions so you can join the results back to the catalogue. The example below also makes the frozen CLS vectors available for reuse.

In [ ]:
preview = working.iloc[:WORKING_PREVIEW_ROWS]
working_inputs, _ = prepare_photometry(preview)
working_loader = make_loader(working_inputs)
working_embeddings = predict(frozen_encoder.encode, working_loader)
frozen_logits = predict(frozen_classifier.eval(), make_loader((working_embeddings,)))
finetuned_logits = predict(finetuned_classifier.eval(), working_loader)

predictions = pd.DataFrame({
    "OBJECT_ID": preview["OBJECT_ID"].to_numpy(),
    "frozen_class": [CLASS_NAMES[i] for i in frozen_logits.argmax(dim=1).tolist()],
    "finetuned_class": [CLASS_NAMES[i] for i in finetuned_logits.argmax(dim=1).tolist()],
})
predictions.head()